# Libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score, classification_report

DIR_DATA = os.getcwd() + "/data/"

# Load Data

In [ ]:
df_normal = pd.read_csv(DIR_DATA + "normal.csv", sep=";", decimal=".")
df_normal['CLASS'] = 0

df_falha1 = pd.read_csv(DIR_DATA + "falha1.csv", sep=";", decimal=".")
df_falha1['CLASS'] = 1

df_falha2 = pd.read_csv(DIR_DATA + "falha2.csv", sep=";", decimal=".")
df_falha2['CLASS'] = 2

# Concatenando os datasets
df_completo = pd.concat([df_normal, df_falha1, df_falha2], ignore_index=True)

X = df_completo.drop(columns=['CLASS']).values
y = df_completo['target'].values

# Divisão de Dados: 60% Treino, 20% Validação, 20% Teste
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Normalização das features (fundamental para descida de gradiente)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Model

In [ ]:
class SoftmaxRegressionGD:
    def __init__(self, lr=0.01, epochs=500):
        self.lr = lr
        self.epochs = epochs
        self.W = None
        self.b = None
        self.history_loss_train = []
        self.history_loss_val = []

    def _softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def _to_one_hot(self, y, num_classes):
        one_hot = np.zeros((len(y), num_classes))
        one_hot[np.arange(len(y)), y] = 1
        return one_hot

    def _compute_loss(self, P, Y_one_hot):
        return -np.mean(np.sum(Y_one_hot * np.log(P + 1e-15), axis=1))

    def fit(self, X_train, y_train, X_val, y_val):
        num_samples, num_features = X_train.shape
        num_classes = len(np.unique(y_train))
        
        # Inicialização de pesos
        self.W = np.zeros((num_features, num_classes))
        self.b = np.zeros((1, num_classes))
        
        Y_train_hot = self._to_one_hot(y_train, num_classes)
        Y_val_hot = self._to_one_hot(y_val, num_classes)

        for epoch in range(self.epochs):
            # Forward pass (Treino)
            Z_train = np.dot(X_train, self.W) + self.b
            P_train = self._softmax(Z_train)
            loss_train = self._compute_loss(P_train, Y_train_hot)
            self.history_loss_train.append(loss_train)

            # Forward pass (Validação)
            Z_val = np.dot(X_val, self.W) + self.b
            P_val = self._softmax(Z_val)
            loss_val = self._compute_loss(P_val, Y_val_hot)
            self.history_loss_val.append(loss_val)

            # Gradientes
            dW = (1 / num_samples) * np.dot(X_train.T, (P_train - Y_train_hot))
            db = (1 / num_samples) * np.sum(P_train - Y_train_hot, axis=0, keepdims=True)

            # Atualização dos parâmetros (Descida de Gradiente)
            self.W -= self.lr * dW
            self.b -= self.lr * db

    def predict(self, X):
        Z = np.dot(X, self.W) + self.b
        P = self._softmax(Z)
        return np.argmax(P, axis=1)

In [ ]:
model = SoftmaxRegressionGD(lr=0.001, epochs=300)
model.fit(X_train, y_train, X_val, y_val)

# Metrics

In [ ]:
def avaliar_conjunto(nome, X_set, y_true):
    y_pred = model.predict(X_set)
    print(f"\n--- Métricas do Conjunto de {nome} ---")
    print(f"Acurácia: {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Sensibilidade/Recall (Macro): {recall_score(y_true, y_pred, average='macro'):.4f}")
    print("\nMatriz de Confusão:")
    print(confusion_matrix(y_true, y_pred))

avaliar_conjunto("Treino", X_train, y_train)
avaliar_conjunto("Validação", X_val, y_val)
avaliar_conjunto("Teste", X_test, y_test)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(model.history_loss_train, label='Perda - Treino', color='blue')
plt.plot(model.history_loss_val, label='Perda - Validação', color='orange', linestyle='--')
plt.title('Evolução da Perda durante o Treinamento (Gradient Descent)')
plt.xlabel('Época')
plt.ylabel('Função de Perda (Cross-Entropy)')
plt.legend()
plt.grid(True)
plt.savefig('evolucao_perda.png')
print("\nGráfico salvo como 'evolucao_perda.png'.")